<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Lysine_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: KO logic for M00016 (Lysine biosynthesis, succinyl-DAP pathway) ===
def check_lysine_module(ko_list):
    status = {}
    status["Aspartate_kinase"] = any(k in ko_list for k in ["K00928","K12524","K12525","K12526"])
    status["Aspartate_semialdehyde_dehydrogenase"] = ("K00133" in ko_list)
    status["HTPA_synthase"] = ("K01714" in ko_list)
    status["HTPA_reductase"] = ("K00215" in ko_list)
    status["Succinyltransferase"] = ("K00674" in ko_list)
    status["Aminotransferase"] = any(k in ko_list for k in ["K00821","K14267"])
    status["Desuccinylase"] = ("K01439" in ko_list)
    status["Epimerase"] = ("K01778" in ko_list)
    status["Decarboxylase"] = any(k in ko_list for k in ["K01586","K12526"])
    complete = all(status.values())
    return complete, status

# === STEP 5: Build lysine pathway model (permissive exchanges) ===
def build_lysine_model():
    model = Model("Lysine_pathway")

    # Metabolites
    asp = Metabolite("C00049_asp_c", compartment="c")
    atp = Metabolite("C00002_atp_c", compartment="c")
    adp = Metabolite("C00008_adp_c", compartment="c")
    asp_phos = Metabolite("C03082_asp_phos_c", compartment="c")
    asp_semialdehyde = Metabolite("C00441_asp_semialdehyde_c", compartment="c")
    pi = Metabolite("C00009_pi_c", compartment="c")
    nadp = Metabolite("C00006_nadp_c", compartment="c")
    nadph = Metabolite("C00005_nadph_c", compartment="c")
    h = Metabolite("C00080_h_c", compartment="c")
    pyruvate = Metabolite("C00022_pyruvate_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    htpa = Metabolite("C20258_htpa_c", compartment="c")
    thdp = Metabolite("C03972_thdp_c", compartment="c")
    nad = Metabolite("C00003_nad_c", compartment="c")
    nadh = Metabolite("C00004_nadh_c", compartment="c")
    succinylcoa = Metabolite("C00091_succinylcoa_c", compartment="c")
    coa = Metabolite("C00010_coa_c", compartment="c")
    succinyl_oxo = Metabolite("C04462_succinyl_oxo_c", compartment="c")
    succinyl_dap = Metabolite("C04421_succinyl_dap_c", compartment="c")
    akg = Metabolite("C00026_akg_c", compartment="c")
    glu = Metabolite("C00025_glu_c", compartment="c")
    succinate = Metabolite("C00042_succinate_c", compartment="c")
    ll_dap = Metabolite("C00666_ll_dap_c", compartment="c")
    meso_dap = Metabolite("C00680_meso_dap_c", compartment="c")
    lys = Metabolite("C00047_lys_c", compartment="c")
    co2 = Metabolite("C00011_co2_c", compartment="c")

    # Reactions
    r00480 = Reaction("R00480_Aspartate_kinase")
    r00480.add_metabolites({asp:-1, atp:-1, adp:1, asp_phos:1})

    r02291 = Reaction("R02291_Aspartate_semialdehyde_dehydrogenase")
    r02291.add_metabolites({asp_semialdehyde:-1, pi:-1, nadp:-1,
                            asp_phos:1, nadph:1, h:1})

    r10147 = Reaction("R10147_HTPA_synthase")
    r10147.add_metabolites({asp_semialdehyde:-1, pyruvate:-1,
                            htpa:1, h2o:1})

    r04198 = Reaction("R04198_HTPA_reductase_NAD")
    r04198.add_metabolites({thdp:-1, nad:-1, h2o:-1,
                            htpa:1, nadh:1, h:1})

    r04199 = Reaction("R04199_HTPA_reductase_NADP")
    r04199.add_metabolites({thdp:-1, nadp:-1, h2o:-1,
                            htpa:1, nadph:1, h:1})

    r04365 = Reaction("R04365_Succinyltransferase")
    r04365.add_metabolites({succinylcoa:-1, thdp:-1, h2o:-1,
                            coa:1, succinyl_oxo:1})

    r04475 = Reaction("R04475_Aminotransferase")
    r04475.add_metabolites({succinyl_dap:-1, akg:-1,
                            succinyl_oxo:1, glu:1})

    r02734 = Reaction("R02734_Desuccinylase")
    r02734.add_metabolites({succinyl_dap:-1, h2o:-1,
                            succinate:1, ll_dap:1})

    r02735 = Reaction("R02735_Epimerase")
    r02735.add_metabolites({ll_dap:-1, meso_dap:1})

    r00451 = Reaction("R00451_Decarboxylase")
    r00451.add_metabolites({meso_dap:-1, lys:1, co2:1})

    model.add_reactions([r00480,r02291,r10147,r04198,r04199,
                         r04365,r04475,r02734,r02735,r00451])

    # Exchanges for all metabolites (permissive style)
    for met in [asp, atp, adp, asp_phos, asp_semialdehyde, pi, nadp, nadph,
                nad, nadh, h, pyruvate, h2o, htpa, thdp, succinylcoa, coa,
                succinyl_oxo, succinyl_dap, akg, glu, succinate, ll_dap,
                meso_dap, lys, co2]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for lysine
    dm_lys = Reaction("DM_C00047_lys_c")
    dm_lys.add_metabolites({lys:-1})
    dm_lys.lower_bound = 0
    dm_lys.upper_bound = 1000
    model.add_reactions([dm_lys])
    model.objective = dm_lys

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_lysine_module(kos)
    flux_value = None
    if complete:
        model = build_lysine_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        **status,
        "Module_complete": complete,
        "Flux_to_Lysine": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("lysine_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("lysine_flux_genomewise.tsv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.6 MB/s eta 0:00:00


Saving FINAL BLAST.tsv to FINAL BLAST.tsv
    Genome  Aspartate_kinase  Aspartate_semialdehyde_dehydrogenase  \
0    ADCAI              True                                  True   
1    AFCAI              True                                  True   
2      BTB              True                                  True   
3      BTQ              True                                  True   
4   BTQVLC              True                                  True   
5     BTZ1              True                                  True   
6    China              True                                  True   
7   China1              True                                  True   
8    MEAM1              True                                  True   
9     PeMo              True                                  True   
10    SiSi              True                                  True   
11      TV              True                                  True   

    HTPA_synthase  HTPA_reductase  Succinyltran

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>